# Subsidiary Data Analysis
Analyze parsed subsidiary data from SEC filings

## 0. Setup

In [13]:
import polars as pl
from pathlib import Path

pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(200)

# Load CSVs
success_csv = Path("../../../../subsidiaries_SUCCESS.csv")
empty_csv = Path("../../../../subsidiaries_EMPTY.csv")

df = pl.read_csv(success_csv, schema_overrides={"Ownership": pl.Float64})
df_empty = pl.read_csv(empty_csv)

print(f"Total rows (success): {len(df):,}")
print(f"Columns: {df.columns}")

# Count unique accessions
unique_accessions = df["Accession"].n_unique()
unique_empty_accessions = df_empty["Accession"].n_unique()

print(f"\nTotal filings processed: {unique_accessions + unique_empty_accessions:,}")
print(f"- Unique accessions (with subsidiaries): {unique_accessions:,}")
print(f"- Unique accessions (empty): {unique_empty_accessions:,}")


df_empty.head(10)


Total rows (success): 47,735
Columns: ['Accession', 'URL', 'SubsidiaryId', 'Subsidiary', 'Jurisdiction', 'NestingLevel', 'ParentName', 'ParentId', 'Ownership', 'Footnotes']

Total filings processed: 466
- Unique accessions (with subsidiaries): 428
- Unique accessions (empty): 38


Accession,URL
str,str
"""`000003799625000013""","""https://www.sec.gov/Archives/edgar/data/37996/000003799625000013/f12312024exhibit21.htm"""
"""`000105955625000025""","""https://www.sec.gov/Archives/edgar/data/1059556/000105955625000025/mco-20241231xexx21.htm"""
"""`000006008625000036""","""https://www.sec.gov/Archives/edgar/data/60086/000006008625000036/exhibit2101-q42024.htm"""
"""`000155115225000020""","""https://www.sec.gov/Archives/edgar/data/1551152/000155115225000020/abbv-20241231xex21.htm"""
"""`000162828025005100""","""https://www.sec.gov/Archives/edgar/data/943452/000162828025005100/wabex210-10k2024.htm"""
"""`000175178825000012""","""https://www.sec.gov/Archives/edgar/data/1751788/000175178825000012/dowinc202410kex21.htm"""
"""`000093646825000009""","""https://www.sec.gov/Archives/edgar/data/936468/000093646825000009/ex21q42024.htm"""
"""`000141344725000019""","""https://www.sec.gov/Archives/edgar/data/1413447/000141344725000019/a211listofsubsidiaries20.htm"""
"""`000107173925000027""","""https://www.sec.gov/Archives/edgar/data/1071739/000107173925000027/a2024123110-kexhibit21.htm"""


In [14]:
# Query rows for a specific accession
accession_to_find = "`000106770125000008"
df.filter(pl.col("Accession") == accession_to_find)

Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000106770125000008""","""https://www.sec.gov/Archives/edgar/data/1067701/000106770125000008/uri-2024123110kex21.htm""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410-sub-1""","""UNITED RENTALS""","""INC. & SUBSIDIARIES""",0,"""UNITED RENTALS, INC.""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410""",null,""""""
"""`000106770125000008""","""https://www.sec.gov/Archives/edgar/data/1067701/000106770125000008/uri-2024123110kex21.htm""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410-sub-2""","""Incorporation""","""Unknown""",0,"""UNITED RENTALS, INC.""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410""",null,""""""
"""`000106770125000008""","""https://www.sec.gov/Archives/edgar/data/1067701/000106770125000008/uri-2024123110kex21.htm""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410-sub-3""","""(a) United Rentals Asia Pacific Holdings Pty Ltd""","""Unknown""",0,"""UNITED RENTALS, INC.""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410""",null,""""""
"""`000106770125000008""","""https://www.sec.gov/Archives/edgar/data/1067701/000106770125000008/uri-2024123110kex21.htm""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410-sub-4""","""B. Harbor Point Insurance Company""","""Unknown""",0,"""UNITED RENTALS, INC.""","""65bf51a8-6f5f-5983-b70a-ee98d81d2410""",null,""""""


In [15]:
# Find rows with empty or null jurisdiction
empty_jurisdiction = df.filter(
    pl.col("Jurisdiction").is_null() | (pl.col("Jurisdiction").str.strip_chars() == "")
)

unique_accessions = empty_jurisdiction.select("Accession").unique()
print(f"Rows with empty jurisdiction: {len(empty_jurisdiction):,}")

print(f"\nUnique accession list({len(unique_accessions)}): ")
print(unique_accessions.to_series().to_list())
empty_jurisdiction.head(10)

Rows with empty jurisdiction: 0

Unique accession list(0): 
[]


Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str


## 1. Nested Subsidiaries (NestingLevel >= 1)

In [16]:
# Find nested subsidiaries
nested = df.filter(pl.col("NestingLevel") >= 1)

nested_accessions = nested.select("Accession").unique()
print(f"Nested subsidiaries (level >= 1): {len(nested):,}")

print(f"Unique accession list({len(nested_accessions)}):")
print(nested_accessions.to_series().to_list())
# nested.head(20)


# Filter by specific accession
pl.Config.set_tbl_rows(-1) 
target_accession = '`000007889025000059'
df.filter(pl.col("Accession") == target_accession)


Nested subsidiaries (level >= 1): 318
Unique accession list(10):
['`000008066125000007', '`000111316925000007', '`000000428125000011', '`000010781525000103', '`000005114325000015', '`000086073125000007', '`000133692025000006', '`000162828025056742', '`000001961725000270', '`000089542125000304']


Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str


In [17]:
# Distribution of nesting levels
df.group_by("NestingLevel").agg(pl.count().alias("count")).sort("NestingLevel")

/var/folders/pg/rrvt9fgx479brk1wjwzw9bqc0000gn/T/ipykernel_34712/269099081.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  df.group_by("NestingLevel").agg(pl.count().alias("count")).sort("NestingLevel")


NestingLevel,count
i64,u32
0,47417
1,267
2,37
3,11
4,3


In [18]:
# TODO: Find parent rows based on ParentId = SubsidiaryId
# (Commented out - we don't have SubsidiaryId column yet)

# nested_with_parents = nested.join(
#     df.select(["SubsidiaryId", "Subsidiary", "Jurisdiction"]).rename({"Subsidiary": "ParentSubsidiary", "Jurisdiction": "ParentJurisdiction"}),
#     left_on="ParentId",
#     right_on="SubsidiaryId",
#     how="left"
# )
# nested_with_parents

## 2. Rows with Footnotes

In [19]:
# Find rows with non-empty footnotes
with_footnotes = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
)

footnotes_accessions = with_footnotes.select("Accession").unique()

print(f"Rows with footnotes: {len(with_footnotes):,}")
print(f"Percentage: {len(with_footnotes) / len(df) * 100:.1f}%")

print(f"\nUnique Accession list({len(footnotes_accessions)}):")
print(footnotes_accessions.to_series().to_list())
with_footnotes.head(20)

Rows with footnotes: 382
Percentage: 0.8%

Unique Accession list(37):
['`000000497725000047', '`000138119725000036', '`000132440425000006', '`000076447825000007', '`000109921925000044', '`000007747625000007', '`000089905125000015', '`000093570325000015', '`000003408825000010', '`000155837025001222', '`000031920125000024', '`000155837025001263', '`000087721225000027', '`000162828025037656', '`000105350725000025', '`000128176125000010', '`000010416925000021', '`000171126925000004', '`000001154425000005', '`000004054525000015', '`000151029525000012', '`000005125325000013', '`000006270925000015', '`000162828025007110', '`000155837025003413', '`000007736025000006', '`000000248825000012', '`000162828025006093', '`000087532025000053', '`000085973725000072', '`000139718725000013', '`000112336025000011', '`000095017025024750', '`000119312525034579', '`000199681025000011', '`000008980025000030', '`000162828025005715']


Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000171126925000004""","""https://www.sec.gov/Archives/edgar/data/1711269/000171126925000004/evrg-12312024xex211.htm""","""f61e0379-7c43-5837-800f-7fce7d76a27d-sub-1""","""Name of CompanyState of IncorporationEvergy Metro, Inc.MissouriEvergy Missouri West, Inc.DelawareEvergy Kansas Central, Inc.KansasEvergy Kansas South""","""Inc. Kansas""",0,"""Evergy, Inc.""","""f61e0379-7c43-5837-800f-7fce7d76a27d""",null,"""2"""
"""`000171126925000004""","""https://www.sec.gov/Archives/edgar/data/1711269/000171126925000004/evrg-12312024xex211.htm""","""f61e0379-7c43-5837-800f-7fce7d76a27d-sub-2""","""Evergy Kansas South""","""Inc.""",0,"""Evergy, Inc.""","""f61e0379-7c43-5837-800f-7fce7d76a27d""",null,"""2"""
"""`000171126925000004""","""https://www.sec.gov/Archives/edgar/data/1711269/000171126925000004/evrg-12312024xex211.htm""","""f61e0379-7c43-5837-800f-7fce7d76a27d-sub-3""","""Evergy Kansas South, Inc. is a subsidiary of Evergy Kansas Central""","""Inc.""",0,"""Evergy, Inc.""","""f61e0379-7c43-5837-800f-7fce7d76a27d""",null,"""2"""
"""`000008980025000030""","""https://www.sec.gov/Archives/edgar/data/89800/000008980025000030/shw-12312024xex211.htm""","""746e8aad-d04c-511c-9a7d-b834fba10378-sub-2""","""1 The Sherwin-Williams Foundation is a 501(c) organization and is not included within the consolidated financial statements of The Sherwin""","""Williams Company.""",0,"""SHERWIN WILLIAMS CO""","""746e8aad-d04c-511c-9a7d-b834fba10378""",null,"""3"""
"""`000162828025006093""","""https://www.sec.gov/Archives/edgar/data/315293/000162828025006093/exhibit2112024.htm""","""18015b3d-9cb8-540c-acbf-c8caf1fcf930""","""Delek Motors Insurance Agency Ltd.""","""Israel""",0,"""Aon plc""","""9dfbce61-e3c6-5b84-950b-fe96ee044019""",null,"""2003"""
"""`000162828025006093""","""https://www.sec.gov/Archives/edgar/data/315293/000162828025006093/exhibit2112024.htm""","""2ebd1f27-7dad-506d-82b9-79963cf1506f""","""I. Beck Insurance Agency Ltd.""","""Israel""",0,"""Aon plc""","""9dfbce61-e3c6-5b84-950b-fe96ee044019""",null,"""1994"""
"""`000000248825000012""","""https://www.sec.gov/Archives/edgar/data/2488/000000248825000012/ex21-10kfy2412_28.htm""","""977c9b79-a2e5-5281-b5f8-7882adce84d8-sub-5""","""Auviz Systems Inc""","""*""",0,"""ADVANCED MICRO DEVICES INC""","""977c9b79-a2e5-5281-b5f8-7882adce84d8""",null,"""1"""
"""`000000248825000012""","""https://www.sec.gov/Archives/edgar/data/2488/000000248825000012/ex21-10kfy2412_28.htm""","""977c9b79-a2e5-5281-b5f8-7882adce84d8-sub-6""","""Xilinx Development Corporation""","""Unknown""",0,"""ADVANCED MICRO DEVICES INC""","""977c9b79-a2e5-5281-b5f8-7882adce84d8""",null,"""1"""
"""`000000248825000012""","""https://www.sec.gov/Archives/edgar/data/2488/000000248825000012/ex21-10kfy2412_28.htm""","""977c9b79-a2e5-5281-b5f8-7882adce84d8-sub-16""","""Level 5 Networks""","""Inc.""",0,"""ADVANCED MICRO DEVICES INC""","""977c9b79-a2e5-5281-b5f8-7882adce84d8""",null,"""2"""


In [20]:
# Unique footnote values per accession
footnotes_by_accession = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
).group_by("Accession").agg(
    pl.col("Footnotes").unique().alias("UniqueFootnotes")
)

print(f"Accessions with footnotes: {len(footnotes_by_accession)}")
print(f"\nAccession: Footnotes")
for row in footnotes_by_accession.iter_rows():
    print(f"{row[0]}: {row[1]}")

Accessions with footnotes: 37

Accession: Footnotes
`000007747625000007: ['2018']
`000109921925000044: ['2002']
`000155837025003413: ['1', '2, 15', '3', '4', '5', '6', '7', '9', '10', '12', '13, 15', '14', '14, 15', '16', '17', '18', '19', '21']
`000007736025000006: ['1', '2', '3', '4', '5', '6']
`000089905125000015: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19']
`000162828025037656: ['1']
`000000497725000047: ['1', '2', '3', '4', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17']
`000155837025001222: ['10']
`000155837025001263: ['1']
`000112336025000011: ['2', '1']
`000105350725000025: ['2', '3', '5', '1', '6', '4']
`000095017025024750: ['2', '3', '4']
`000031920125000024: ['1992', '1987', '2002']
`000139718725000013: ['1']
`000119312525034579: ['1', '2']
`000093570325000015: ['1']
`000076447825000007: ['1', '2', '3', '4', '5', '6', '7']
`000171126925000004: ['2']
`000138119725000036: ['1', '2', '3']
`00000024

In [21]:
# Unique footnote values - extract all individual numbers with their accessions
import re

footnotes_with_accession = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
).select(["Accession", "Footnotes"]).unique()

# Build a dict: number -> list of accessions
number_to_accessions = {}
for row in footnotes_with_accession.iter_rows():
    accession, footnote = row
    nums = re.findall(r'\d+', footnote)
    for n in nums:
        num = int(n)
        if num not in number_to_accessions:
            number_to_accessions[num] = set()
        number_to_accessions[num].add(accession)

# Sort by number descending and print
sorted_items = sorted(number_to_accessions.items(), key=lambda x: x[0], reverse=True)

print(f"Total unique footnote numbers: {len(sorted_items)}")
print(f"\nFootnoteNumber: Accessions")
for num, accs in sorted_items:
    print(f"{num}: {list(accs)}")

Total unique footnote numbers: 38

FootnoteNumber: Accessions
2018: ['`000007747625000007']
2007: ['`000087721225000027']
2005: ['`000006270925000015']
2003: ['`000162828025006093']
2002: ['`000109921925000044', '`000031920125000024']
1999: ['`000006270925000015']
1994: ['`000162828025006093']
1992: ['`000031920125000024']
1987: ['`000031920125000024']
43: ['`000162828025005715']
29: ['`000000248825000012']
28: ['`000000248825000012']
27: ['`000000248825000012']
26: ['`000000248825000012']
25: ['`000000248825000012']
24: ['`000000248825000012']
23: ['`000162828025005715']
22: ['`000000248825000012']
21: ['`000085973725000072', '`000155837025003413', '`000000248825000012']
19: ['`000155837025003413', '`000089905125000015']
18: ['`000000248825000012', '`000155837025003413', '`000089905125000015']
17: ['`000000497725000047', '`000000248825000012', '`000155837025003413', '`000089905125000015']
16: ['`000000497725000047', '`000155837025003413', '`000089905125000015', '`000000248825000012']
